# Advanced OOP

This notebook explores advanced Object-Oriented Programming concepts in Python. We will go beyond basic classes and inheritance to look at magic methods, method types, property decorators, method resolution order (MRO), and modern dataclasses.

## 1. Magic (Dunder) Methods and Operator Overloading

Magic methods (or "dunder" methods, for double-underscore) allow you to define how your objects behave with built-in Python operations like addition (`+`), printing, and length checking (`len()`).

In [1]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # Controls what developers see (debugging)
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    # Controls what users see (printing)
    def __str__(self):
        return f"({self.x}i + {self.y}j)"

    # Operator Overloading for addition (+)
    def __add__(self, other):
        if not isinstance(other, Vector):
            raise TypeError("Can only add two Vectors")
        return Vector(self.x + other.x, self.y + other.y)

    # Define what boolean value the object evaluates to
    def __bool__(self):
        return bool(self.x or self.y)

v1 = Vector(2, 4)
v2 = Vector(3, -1)
v3 = v1 + v2

print("Repr:", repr(v1))
print("Str:", str(v1))
print("Addition:", v3)
print("Is v1 truthy?", bool(v1))
print("Is Vector(0,0) truthy?", bool(Vector(0, 0)))

Repr: Vector(2, 4)
Str: (2i + 4j)
Addition: (5i + 3j)
Is v1 truthy? True
Is Vector(0,0) truthy? False


## 2. Class Methods vs. Static Methods

- **Regular methods** take `self` as the first argument and operate on instance data.
- **`@classmethod`** takes `cls` as the first argument. It operates on the class itself, not instances. Often used as "factory methods" to create instances in alternative ways.
- **`@staticmethod`** takes neither `self` nor `cls`. It acts like a regular function that just happens to live inside a class's namespace for organizational purposes.

In [2]:
import datetime

class Employee:
    raise_amount = 1.05  # Class variable

    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    # 1. Class Method (Factory)
    @classmethod
    def from_string(cls, employee_str):
        name, salary = employee_str.split('-')
        return cls(name, float(salary))

    # 2. Static Method (Utility)
    @staticmethod
    def is_workday(day):
        # 5 = Saturday, 6 = Sunday
        if day.weekday() == 5 or day.weekday() == 6:
            return False
        return True

# Using the class method to create an instance
emp_str = "John-70000"
emp1 = Employee.from_string(emp_str)
print(f"Created: {emp1.name} with salary ${emp1.salary}")

# Using the static method
today = datetime.date.today()
print(f"Is today a workday? {Employee.is_workday(today)}")

Created: John with salary $70000.0
Is today a workday? True


## 3. Encapsulation: The `@property` Decorator

Python doesn't have true "private" variables, but it has conventions (prefixing with `_`). To control getting and setting attributes without breaking a user's code, we use `@property` to create getters, setters, and deleters.

In [3]:
class Temperature:
    def __init__(self, celsius):
        # Use the setter immediately upon initialization
        self.celsius = celsius

    # Getter
    @property
    def celsius(self):
        return self._celsius

    # Setter
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero is not possible.")
        self._celsius = value

    # Derived property (computed on the fly)
    @property
    def fahrenheit(self):
        return (self.celsius * 9 / 5) + 32

temp = Temperature(25)
print("Celsius:", temp.celsius)
print("Fahrenheit:", temp.fahrenheit)

# This uses the setter validation
temp.celsius = 30
print("New Fahrenheit:", temp.fahrenheit)

# Uncommenting below raises a ValueError
# temp.celsius = -300

Celsius: 25
Fahrenheit: 77.0
New Fahrenheit: 86.0


## 4. Multiple Inheritance and MRO

Python supports multiple inheritance. When a class inherits from multiple classes that share method names, Python uses the **Method Resolution Order (MRO)** to determine which method to call. It uses the C3 Linearization algorithm.

In [4]:
class A:
    def process(self):
        print("A processing")

class B(A):
    def process(self):
        print("B processing")
        super().process()

class C(A):
    def process(self):
        print("C processing")
        super().process()

# The Diamond Problem
class D(B, C):
    def process(self):
        print("D processing")
        super().process()

d = D()
d.process()

print("\nMethod Resolution Order (MRO):")
for cls in D.__mro__:
    print(" -", cls.__name__)

D processing
B processing
C processing
A processing

Method Resolution Order (MRO):
 - D
 - B
 - C
 - A
 - object


## 5. Modern Python: Dataclasses

Introduced in Python 3.7, the `@dataclass` decorator automatically generates boilerplate methods like `__init__`, `__repr__`, and `__eq__` for classes that primarily exist to store data.

In [5]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Product:
    name: str
    price: float
    quantity: int = 0
    # field(default_factory=...) is used for mutable defaults like lists
    tags: List[str] = field(default_factory=list)

    def total_value(self) -> float:
        return self.price * self.quantity

# Auto-generated __init__
p1 = Product("Laptop", 1200.00, 5, ["electronics", "work"])
p2 = Product("Laptop", 1200.00, 5, ["electronics", "work"])

# Auto-generated __repr__
print(p1)

# Auto-generated __eq__ (checks values, not memory address)
print("Are p1 and p2 equal?", p1 == p2)

# Custom method still works
print("Total Value:", p1.total_value())

Product(name='Laptop', price=1200.0, quantity=5, tags=['electronics', 'work'])
Are p1 and p2 equal? True
Total Value: 6000.0
